# FeatureGraph BIDMC respiration representation study

This notebook is the sole executable workflow for the first-paper BIDMC study. The FeatureGraph construction is written directly from observations through states, events, plateau intervals, and objects. SciPy appears only in the frozen comparator path. No subject-specific tuning is performed.


## Complete study implementation

Frozen parameters: BIDMC 1.0.0, subjects 1–53, 125 Hz, max–then–mean window 100, numerical absolute tolerance 1e-12, floor-midpoint projection of numerically flat extrema plateaus, and 63-sample ordered matching tolerance.


In [ ]:
from __future__ import annotations

from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
from scipy.signal import butter, find_peaks, sosfiltfilt

DATASET_VERSION = "1.0.0"
BASE = f"https://physionet.org/files/bidmc/{DATASET_VERSION}/bidmc_csv"
CACHE = Path(".bidmc_notebook_cache")
FS = 125
W = 100
TOL = 63
NUMERICAL_ATOL = 1e-12


def load(subject: int, kind: str = "Signals") -> pd.DataFrame:
    CACHE.mkdir(exist_ok=True)
    path = CACHE / f"bidmc_{subject:02d}_{kind}.csv"
    if not path.exists():
        urlretrieve(f"{BASE}/{path.name}", path)
    frame = pd.read_csv(path)
    frame.columns = frame.columns.str.strip()
    return frame


def exact_runs(signal: pd.Series):
    # Treat floating-point residue below the construction tolerance as
    # part of the same numerically flat run.
    run_id = signal.diff().abs().gt(NUMERICAL_ATOL).cumsum()
    bounds = (
        pd.DataFrame({"sample_index": signal.index, "run_id": run_id})
        .groupby("run_id", sort=False)["sample_index"]
        .agg(["min", "max"])
    )
    return run_id, bounds


def project_run_midpoint(anchor: int, run_id, bounds):
    current = run_id.at[anchor]
    start = int(bounds.at[current, "min"])
    end = int(bounds.at[current, "max"])
    return start, start + (end - start) // 2, end


def construct(subject: int, window: int = W):
    source = load(subject)
    raw = source["RESP"].astype(float)
    df = pd.DataFrame({
        "subject_id": subject,
        "sample_index": raw.index,
        "time_seconds": raw.index / FS,
        "respiration": raw,
    })
    df["respiration_smooth"] = (
        raw.rolling(window, min_periods=window)
        .max()
        .rolling(window, min_periods=window)
        .mean()
        .shift(-window)
    )
    # Stateful construction is evaluated only where the envelope exists.
    # This prevents the rolling-window edge from creating a false exit event.
    prepared = df.loc[df["respiration_smooth"].notna()].copy()
    prepared["respiration_change"] = prepared["respiration_smooth"].diff()
    prepared["respiration_smooth_valid"] = prepared["respiration_change"].notna()
    # NUMERICAL_ATOL separates floating-point residue from directional
    # change. It is not a physiological amplitude threshold.
    prepared["respiration_rising"] = prepared["respiration_change"].gt(NUMERICAL_ATOL)
    prepared["respiration_falling"] = prepared["respiration_change"].lt(-NUMERICAL_ATOL)
    prepared["respiration_inactive"] = prepared["respiration_change"].abs().le(NUMERICAL_ATOL)
    prepared["enter_respiration_rising"] = prepared["respiration_rising"].astype(int).diff().eq(1)
    prepared["exit_respiration_rising"] = prepared["respiration_rising"].astype(int).diff().eq(-1)
    prepared["exit_respiration_rising_id"] = prepared["exit_respiration_rising"].cumsum()
    prepared["enter_respiration_rising_id"] = prepared["enter_respiration_rising"].cumsum()
    for column in [
        "respiration_change", "respiration_smooth_valid", "respiration_rising",
        "respiration_falling", "respiration_inactive", "enter_respiration_rising",
        "exit_respiration_rising", "exit_respiration_rising_id",
        "enter_respiration_rising_id",
    ]:
        df[column] = prepared[column]
    state_count = prepared[[
        "respiration_rising", "respiration_falling", "respiration_inactive"
    ]].sum(axis=1)
    assert state_count[prepared["respiration_smooth_valid"]].eq(1).all()

    # Directional states describe the edge ending at the current row, so the
    # extremum event is assigned to the preceding sample.
    prepared["peak_event"] = prepared["exit_respiration_rising"].shift(-1, fill_value=False)
    prepared["trough_event"] = prepared["enter_respiration_rising"].shift(-1, fill_value=False)
    prepared["peak_event_index"] = pd.Series(
        np.where(prepared["peak_event"], prepared.index, np.nan), index=prepared.index
    ).ffill()
    prepared["trough_event_index"] = pd.Series(
        np.where(prepared["trough_event"], prepared.index, np.nan), index=prepared.index
    ).ffill()

    grouped = prepared.groupby("enter_respiration_rising_id", sort=False)
    objects = grouped.agg(
        start_index=("trough_event_index", "first"),
        peak_index=("peak_event_index", "max"),
        end_index=("trough_event_index", "max"),
        has_start=("enter_respiration_rising", "max"),
        raw_minimum=("respiration", "min"),
        raw_maximum=("respiration", "max"),
    ).reset_index().rename(columns={"enter_respiration_rising_id": "featuregraph_object_id"})
    last_id = objects["featuregraph_object_id"].max()
    objects["transition_complete"] = (
        objects["has_start"].astype(bool)
        & objects["start_index"].notna()
        & objects["peak_index"].notna()
        & objects["end_index"].notna()
        & objects["start_index"].lt(objects["peak_index"])
        & objects["peak_index"].lt(objects["end_index"])
        & objects["featuregraph_object_id"].lt(last_id)
    )

    run_id, bounds = exact_runs(prepared["respiration_smooth"])
    boundary_specs = ["start_index", "peak_index", "end_index"]
    interval_names = ["start_trough", "peak", "end_trough"]
    for boundary, prefix in zip(boundary_specs, interval_names):
        starts, midpoints, ends = [], [], []
        for anchor in objects[boundary]:
            if pd.isna(anchor):
                starts.append(np.nan); midpoints.append(np.nan); ends.append(np.nan)
            else:
                start, midpoint, end = project_run_midpoint(int(anchor), run_id, bounds)
                starts.append(start); midpoints.append(midpoint); ends.append(end)
        objects[f"{prefix}_start_index"] = starts
        objects[f"{prefix}_end_index"] = ends
        objects[boundary] = midpoints

    intervals_present = objects[[
        "start_trough_end_index", "peak_start_index", "peak_end_index",
        "end_trough_start_index",
    ]].notna().all(axis=1)
    intervals_ordered = (
        objects["start_trough_end_index"].lt(objects["peak_start_index"])
        & objects["peak_end_index"].lt(objects["end_trough_start_index"])
    ).fillna(False)
    objects["plateau_boundary_ambiguous"] = intervals_present & ~intervals_ordered
    objects["plateau_invalidated_complete"] = objects["transition_complete"] & objects["plateau_boundary_ambiguous"]
    objects["is_complete"] = objects["transition_complete"] & intervals_ordered
    objects["period_seconds"] = objects["peak_index"].diff() / FS
    duration = objects["end_index"] - objects["start_index"]
    objects["temporal_symmetry"] = (
        1 - ((objects["peak_index"] - objects["start_index"]) -
             (objects["end_index"] - objects["peak_index"])).abs() / duration
    ).where(duration > 0)
    objects["full_excursion"] = objects["raw_maximum"] - objects["raw_minimum"]
    objects["subject"] = subject
    objects["peak_detection_index"] = objects["peak_end_index"] + window
    objects["peak_detection_latency_samples"] = objects["peak_detection_index"] - objects["peak_index"]

    detected = []
    for anchor in prepared.index[prepared["peak_event"]]:
        detected.append(project_run_midpoint(int(anchor), run_id, bounds)[1])
    invalidated = objects.loc[objects["plateau_invalidated_complete"]].copy()
    return df, objects, invalidated, detected


def baseline(subject: int):
    raw = load(subject)["RESP"].astype(float).to_numpy()
    sos = butter(4, 0.8, btype="lowpass", fs=FS, output="sos")
    filtered = sosfiltfilt(sos, raw)
    peaks, _ = find_peaks(filtered, distance=188, prominence=0.08)
    troughs, _ = find_peaks(-filtered, distance=188, prominence=0.08)
    rows = []
    for start, end in zip(troughs[:-1], troughs[1:]):
        candidates = peaks[(peaks > start) & (peaks < end)]
        if len(candidates) != 1:
            continue
        peak = int(candidates[0])
        preceding = peaks[peaks < peak]
        duration = int(end - start)
        interval = raw[int(start):int(end) + 1]
        rows.append({
            "llm_object_id": len(rows) + 1,
            "start_index": int(start), "peak_index": peak, "end_index": int(end),
            "is_complete": True,
            "period_seconds": ((peak - int(preceding[-1])) / FS if len(preceding) else np.nan),
            "full_excursion": float(interval.max() - interval.min()),
            "temporal_symmetry": 1 - abs(
                (peak - int(start)) - (int(end) - peak)
            ) / duration,
        })
    return pd.DataFrame(rows), peaks.astype(int).tolist()


def optimal_pairs(left_values, right_values, tolerance=TOL):
    left_count, right_count = len(left_values), len(right_values)
    matched = np.zeros((left_count + 1, right_count + 1), dtype=int)
    error = np.zeros((left_count + 1, right_count + 1), dtype=float)
    choice = np.zeros((left_count + 1, right_count + 1), dtype=np.int8)
    for i in range(1, left_count + 1):
        for j in range(1, right_count + 1):
            candidates = [
                (matched[i - 1, j], error[i - 1, j], 1),
                (matched[i, j - 1], error[i, j - 1], 2),
            ]
            distance = abs(left_values[i - 1] - right_values[j - 1])
            if distance <= tolerance:
                candidates.append((matched[i - 1, j - 1] + 1, error[i - 1, j - 1] + distance, 3))
            best = max(candidates, key=lambda item: (item[0], -item[1], item[2]))
            matched[i, j], error[i, j], choice[i, j] = best
    pairs, i, j = [], left_count, right_count
    while i and j:
        if choice[i, j] == 3:
            pairs.append((i - 1, j - 1)); i -= 1; j -= 1
        elif choice[i, j] == 1:
            i -= 1
        else:
            j -= 1
    return list(reversed(pairs))


def compare_objects(featuregraph, comparator):
    left = featuregraph.loc[featuregraph["is_complete"]].sort_values("peak_index").reset_index(drop=True)
    right = comparator.loc[comparator["is_complete"]].sort_values("peak_index").reset_index(drop=True)
    pairs = optimal_pairs(left.peak_index.tolist(), right.peak_index.tolist())
    li, ri = {x for x, _ in pairs}, {y for _, y in pairs}
    rows = []
    for x, y in pairs:
        row = {"subject": int(left.at[x, "subject"]), "featuregraph_object_id": left.at[x, "featuregraph_object_id"], "llm_object_id": right.at[y, "llm_object_id"]}
        for column in ["start_index", "peak_index", "end_index", "period_seconds", "full_excursion", "temporal_symmetry"]:
            row[f"featuregraph_{column}"] = left.at[x, column]
            row[f"llm_{column}"] = right.at[y, column]
            row[f"delta_{column}"] = left.at[x, column] - right.at[y, column]
        rows.append(row)
    matched = pd.DataFrame(rows)
    return (
        matched,
        left.loc[~left.index.isin(li)].copy().reset_index(drop=True),
        right.loc[~right.index.isin(ri)].copy().reset_index(drop=True),
    )


def annotation_status(subject, detected):
    annotations = load(subject, "Breaths")
    rows, unmatched_by_annotator = [], {}
    for column in ["breaths ann1 [signal sample no]", "breaths ann2 [signal sample no]"]:
        annotator = column.split()[1]
        reference = annotations[column].dropna().astype(int).sort_values().tolist()
        pairs = optimal_pairs(detected, reference)
        di, ri = {x for x, _ in pairs}, {y for _, y in pairs}
        unmatched_by_annotator[annotator] = {detected[i] for i in range(len(detected)) if i not in di}
        rows.append({
            "subject": subject, "annotator": annotator,
            "detected_peaks": len(detected), "reference_peaks": len(reference),
            "matched": len(pairs),
            "matched_fraction_detected": len(pairs) / len(detected),
            "matched_fraction_reference": len(pairs) / len(reference),
        })
    return pd.DataFrame(rows), unmatched_by_annotator


def run_subject(subject: int, window: int = W):
    df, fg, invalidated, detected = construct(subject, window)
    comparator, _ = baseline(subject)
    matched, fg_only, baseline_only = compare_objects(fg, comparator)
    annotation, unmatched = annotation_status(subject, detected)
    if len(fg_only):
        fg_only["excluded_by_ann1"] = fg_only.peak_index.isin(unmatched["ann1"])
        fg_only["excluded_by_ann2"] = fg_only.peak_index.isin(unmatched["ann2"])
        fg_only["excluded_by_both_annotators"] = fg_only.excluded_by_ann1 & fg_only.excluded_by_ann2
    complete_count = int(fg["is_complete"].sum())
    ambiguous_count = int(fg["plateau_boundary_ambiguous"].sum())
    summary = {
        "subject": subject, "samples": len(df),
        "featuregraph_detected_peaks": len(detected),
        "featuregraph_complete_objects": complete_count,
        "baseline_complete_objects": len(comparator),
        "matched_objects": len(matched),
        "featuregraph_only_objects": len(fg_only),
        "baseline_only_objects": len(baseline_only),
        "featuregraph_invalidated_complete_objects": len(invalidated),
        "featuregraph_ambiguous_objects": ambiguous_count,
        "featuregraph_matched_fraction": len(matched) / complete_count if complete_count else np.nan,
        "baseline_matched_fraction": len(matched) / len(comparator) if len(comparator) else np.nan,
    }
    return summary, matched, fg_only, baseline_only, invalidated, annotation


def run_all():
    results = {}
    with ThreadPoolExecutor(max_workers=8) as executor:
        futures = {executor.submit(run_subject, subject): subject for subject in range(1, 54)}
        for future in as_completed(futures):
            subject = futures[future]
            results[subject] = future.result()
            print(f"Completed BIDMC subject {subject:02d}", flush=True)
    ordered = [results[s] for s in range(1, 54)]
    return (
        pd.DataFrame([r[0] for r in ordered]),
        pd.concat([r[1] for r in ordered], ignore_index=True),
        pd.concat([r[2] for r in ordered], ignore_index=True),
        pd.concat([r[3] for r in ordered], ignore_index=True),
        pd.concat([r[4] for r in ordered], ignore_index=True),
        pd.concat([r[5] for r in ordered], ignore_index=True),
    )


## Execute the subject-1 record, full cohort, and sensitivity check


In [ ]:
# Subject 1 development record
subject_1_df, subject_1_objects, subject_1_invalidated, subject_1_peaks = construct(1)
subject_1_summary, subject_1_matched, subject_1_only, subject_1_baseline_only, _, subject_1_annotations = run_subject(1)
subject_1_results = pd.DataFrame([{**subject_1_summary,
    "median_absolute_rate_error_bpm": (60 / subject_1_matched["featuregraph_period_seconds"] - 60 / subject_1_matched["llm_period_seconds"]).abs().median(),
    "median_absolute_period_error_seconds": subject_1_matched["delta_period_seconds"].abs().median(),
    "median_absolute_full_excursion_error": subject_1_matched["delta_full_excursion"].abs().median(),
    "median_absolute_temporal_symmetry_error": subject_1_matched["delta_temporal_symmetry"].abs().median(),
}])
subject_1_results

# Frozen 53-subject study
subject_summary, matched_objects, featuregraph_only_objects, baseline_only_objects, invalidated_objects, annotation_summary = run_all()
matched_objects["featuregraph_rate_bpm"] = 60 / matched_objects["featuregraph_period_seconds"]
matched_objects["baseline_rate_bpm"] = 60 / matched_objects["llm_period_seconds"]
matched_objects["delta_rate_bpm"] = matched_objects["featuregraph_rate_bpm"] - matched_objects["baseline_rate_bpm"]
annotation_totals = annotation_summary.groupby("annotator").agg(detected_peaks=("detected_peaks", "sum"), reference_peaks=("reference_peaks", "sum"), matched=("matched", "sum"))
annotation_totals["detected_matched_fraction"] = annotation_totals["matched"] / annotation_totals["detected_peaks"]
annotation_totals["reference_matched_fraction"] = annotation_totals["matched"] / annotation_totals["reference_peaks"]
cohort_summary = pd.DataFrame([{
    "subjects": subject_summary["subject"].nunique(), "failures": 0,
    "featuregraph_detected_peaks": int(subject_summary["featuregraph_detected_peaks"].sum()),
    "featuregraph_complete_objects": int(subject_summary["featuregraph_complete_objects"].sum()),
    "baseline_complete_objects": int(subject_summary["baseline_complete_objects"].sum()),
    "matched_objects": len(matched_objects), "featuregraph_only_objects": len(featuregraph_only_objects),
    "baseline_only_objects": len(baseline_only_objects),
    "featuregraph_ambiguous_objects": int(subject_summary["featuregraph_ambiguous_objects"].sum()),
    "featuregraph_invalidated_complete_objects": len(invalidated_objects),
    "median_subject_featuregraph_matched_fraction": subject_summary["featuregraph_matched_fraction"].median(),
    "median_subject_baseline_matched_fraction": subject_summary["baseline_matched_fraction"].median(),
    "median_absolute_peak_error_samples": matched_objects["delta_peak_index"].abs().median(),
    "p90_absolute_peak_error_samples": matched_objects["delta_peak_index"].abs().quantile(0.9),
    "median_absolute_rate_error_bpm": matched_objects["delta_rate_bpm"].abs().median(),
    "median_absolute_period_error_seconds": matched_objects["delta_period_seconds"].abs().median(),
    "median_absolute_full_excursion_error": matched_objects["delta_full_excursion"].abs().median(),
    "median_absolute_temporal_symmetry_error": matched_objects["delta_temporal_symmetry"].abs().median(),
    "featuregraph_only_excluded_by_both_annotators": int(featuregraph_only_objects["excluded_by_both_annotators"].sum()),
    "featuregraph_only_retained_by_one_or_both_annotators": int(len(featuregraph_only_objects) - featuregraph_only_objects["excluded_by_both_annotators"].sum()),
}])
cohort_summary.T

# Subject 1 fixed-window sensitivity check
sensitivity_rows = []
for candidate_window in (75, 100, 125):
    summary, pairs, fg_only, comparator_only, invalidated, _ = run_subject(1, candidate_window)
    sensitivity_rows.append({
        "smooth_window_samples": candidate_window, "effective_support_samples": 2 * candidate_window - 1,
        "detected_peaks": summary["featuregraph_detected_peaks"], "complete_objects": summary["featuregraph_complete_objects"],
        "matched_objects": summary["matched_objects"], "featuregraph_only_objects": summary["featuregraph_only_objects"],
        "baseline_only_objects": summary["baseline_only_objects"], "invalidated_objects": len(invalidated),
        "median_absolute_rate_error_bpm": (60 / pairs["featuregraph_period_seconds"] - 60 / pairs["llm_period_seconds"]).abs().median(),
        "median_absolute_period_error_seconds": pairs["delta_period_seconds"].abs().median(),
        "median_absolute_full_excursion_error": pairs["delta_full_excursion"].abs().median(),
        "median_absolute_temporal_symmetry_error": pairs["delta_temporal_symmetry"].abs().median(),
    })
window_sensitivity = pd.DataFrame(sensitivity_rows)
subject_discordance = subject_summary[["subject", "featuregraph_only_objects"]].sort_values("featuregraph_only_objects", ascending=False).reset_index(drop=True)


## Frozen regression contract and report


In [ ]:
expected = {"subjects": 53, "failures": 0, "featuregraph_detected_peaks": 7988, "featuregraph_complete_objects": 7926, "baseline_complete_objects": 7168, "matched_objects": 7086, "featuregraph_only_objects": 840, "baseline_only_objects": 82, "featuregraph_ambiguous_objects": 90, "featuregraph_invalidated_complete_objects": 37, "featuregraph_only_excluded_by_both_annotators": 474, "featuregraph_only_retained_by_one_or_both_annotators": 366}
observed = cohort_summary.iloc[0][list(expected)].to_dict()
assert observed == expected, {key: (expected[key], observed[key]) for key in expected if expected[key] != observed[key]}
assert subject_summary["subject"].tolist() == list(range(1, 54))
assert len(featuregraph_only_objects) == 840
assert len(baseline_only_objects) == 82
subject_13_debug_df, _, _, _ = construct(13)
subject_13_debug_region = subject_13_debug_df.loc[33400:33600]
assert int(subject_13_debug_region["enter_respiration_rising"].sum()) == 1
assert int(subject_13_debug_region["exit_respiration_rising"].sum()) == 2
residue = subject_13_debug_region["respiration_change"].abs().le(NUMERICAL_ATOL)
assert not subject_13_debug_region.loc[residue, ["respiration_rising", "respiration_falling"]].any(axis=None)
print("All frozen BIDMC regression checks passed.")
print("\nCohort summary\n", cohort_summary.T.to_string())
print("\nSubject 1 window sensitivity\n", window_sensitivity.to_string(index=False))
print("\nLargest FeatureGraph-only concentrations\n", subject_discordance.head(15).to_string(index=False))
print("\nAnnotation summary\n", annotation_totals.to_string())
